# 06. List-level reward model and Next-K generation (slate study, level 2)

Every reward of stages 4–6 scores one track as if it were the only one, while the user gets a list. Here (`recsys_rl.slate_rm`, `recsys_rl.nextk`, report §9):

1. **Slate RM** on real lists: runs of consecutive recommendations (`is_organic = 0`) within a session, cut into chunks of 10. It predicts the response to position j from the context and the tracks at positions 1..j (without their feedback — what is known when the whole list is chosen). It starts as the item-level RM applied to every position (a zero-initialised list correction on top of the frozen item-level RM), so its gain over the item-level RM on the val / test lists is what the list structure adds. Proxy = on top of `rm_proxy`, gold = on top of `rm_gold` (audio, other seed, 80% of the users).
2. **Next-K policy**: GPT-2 picks the list item by item, each chosen track appended to the context (repeats masked). RLOO over 8 lists per context with the list reward of the proxy slate RM (mean expected ratio over the 10 positions), KL to the Next-K π_ref summed over the positions. The Top-K variant (`rl.nextk_append=false`) samples the list from one softmax (Plackett–Luce).
3. **Baselines without training**: π_ref Top-K and Next-K, the item-level reranker (the best list of stages 4–5), the per-position tilted greedy search by the slate RM (it ignores the effect of a position on the later ones, which RL does not) and Best-of-16 lists of π_ref.

Needs `runs/base`, `rm_proxy`, `rm_gold` on Drive and `raw/artist_item_mapping.parquet`. Every run resumes on a restart; per-user tables are cached in `runs/nextk_eval/`.

In [ ]:
# Bootstrap: each Colab notebook runs in its own runtime, so repeat the setup from 00.
# Data must already be on Drive (00_colab_setup.ipynb downloads it). Locally: cd to repo root.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
    token = (userdata.get("GH_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
    repo_url = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"
    !git clone -q {repo_url} /content/repo || git -C /content/repo pull -q
    if not Path("/content/repo/pyproject.toml").exists():
        raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
    %cd /content/repo
    !pip install -q uv
    !uv pip install --system -q -e .
    sys.path.insert(0, "/content/repo/src")  # the editable .pth is only read at startup
elif not Path("configs").exists():
    os.chdir(Path.cwd().parent)

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import torch

from recsys_rl.config import load_config, parse_overrides
from recsys_rl.data import prepare
from recsys_rl.nextk import logged_list_effects, nextk_per_user, nextk_reward_scale, reranker_lists
from recsys_rl.rl import item_features, train_rl
from recsys_rl.significance import bootstrap_table
from recsys_rl.slate_rm import load_slate_rm, slate_rm_eval, train_slate_rm, val_slates
from recsys_rl.utils import get_device, save_figure

pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_float_precision(4)

device = get_device()
print(device, torch.cuda.get_device_name() if device.type == "cuda" else "")
cfg = load_config("configs/base.yaml")
prepared = prepare(cfg)
store, bounds = prepared.store.in_memory(), prepared.bounds
RESULTS = cfg.results_dir / "nextk"  # CSV / PNG for the report, on Drive in Colab
RESULTS.mkdir(parents=True, exist_ok=True)
runs = cfg.run_dir.parent


def config(run_name: str, *overrides: str):
    return load_config("configs/base.yaml", **parse_overrides([f"run_name={run_name}", *overrides]))

## Slate RM

Training: 20k steps of 256 logged slates (early stopping on the val-day slates, at most 50k of them). Step 0 of the log is the item-level RM itself (list correction = 0).

In [ ]:
SLATE_RMS = {
    "slate_rm_proxy": ["slate_rm.base_rm=rm_proxy"],
    "slate_rm_gold": ["slate_rm.base_rm=rm_gold", "seed=7", "slate_rm.user_fraction=0.8"],
}
for name, overrides in SLATE_RMS.items():
    train_slate_rm(config(name, *overrides), prepared, device)
pl.concat(
    [pl.read_ndjson(runs / n / "log.jsonl").with_columns(run=pl.lit(n)) for n in SLATE_RMS],
    how="diagonal_relaxed",
).select(
    "run", "step", "loss", "val_slate_loss", "val_item_loss", "val_slate_ratio_spearman",
    "val_item_ratio_spearman", "val_slate_list_spearman", "val_item_list_spearman",
)  # fmt: skip

### Quality on the val and test lists

`slate_*` = slate RM, `item_*` = the item-level RM it extends (same context state, every position scored alone). `list_spearman` = rank correlation of the list reward with the real mean played ratio of the list. Per position: does the gain come from the later positions (where the earlier tracks of the list matter)? The descriptive table checks two list effects in the logs directly: the played ratio by position and by whether the track's artist already occurred earlier in the same list.

In [ ]:
_, _, item_artist = item_features(prepared.items, len(prepared.items) + 2, cfg.raw_dir)
c = cfg.slate_rm
quality, positions, effects = [], [], []
for split, (start, end) in {
    "val": (bounds.val_start, bounds.val_end),
    "test": (bounds.test_start, 2**32),
}.items():
    rows, lengths = val_slates(store, start, end, bounds.train_end, c, cfg.seed)
    cutoff = bounds.val_end if split == "val" else int(store.columns["timestamp"].max()) + 1
    for name in SLATE_RMS:
        model = load_slate_rm(runs / name / "best.pt", device)
        summary, by_pos = slate_rm_eval(
            model, store, rows, lengths, cutoff, cfg.data, cfg.rm.skip_threshold,
            amp=torch.bfloat16 if device.type == "cuda" else None,
        )  # fmt: skip
        quality.append({"split": split, "rm": name, **summary})
        positions.append(by_pos.with_columns(split=pl.lit(split), rm=pl.lit(name)))
    effects.append(
        logged_list_effects(store, rows, lengths, item_artist).with_columns(split=pl.lit(split))
    )
quality = pl.DataFrame(quality)
positions = pl.concat(positions)
effects = pl.concat(effects)
quality.write_csv(RESULTS / "slate_rm_quality.csv")
positions.write_csv(RESULTS / "slate_rm_positions.csv")
effects.write_csv(RESULTS / "list_effects.csv")
quality

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pos = positions.filter((pl.col("split") == "val") & (pl.col("rm") == "slate_rm_proxy"))
axes[0].plot(pos["position"], pos["mean_ratio"], "o-")
axes[0].set(xlabel="position in the logged list", ylabel="real mean played ratio", title="val")
for name in ("slate", "item"):
    axes[1].plot(pos["position"], pos[f"{name}_listen_plus_auc"], "o-", label=f"{name} RM")
    axes[2].plot(pos["position"], pos[f"{name}_ratio_mae"], "o-", label=f"{name} RM")
axes[1].set(xlabel="position", title="Listen+ AUC (proxy, val)")
axes[2].set(xlabel="position", title="ratio MAE (proxy, val)")
axes[1].legend()
save_figure(fig, RESULTS / "slate_rm_positions.png")
plt.show()
effects.filter(pl.col("split") == "val")

## Scale of the list reward

In-group std of the proxy list reward over 8 lists of π_ref per training context (Next-K and Top-K lists). The KL weight is set in units of it, as for the item-level rewards (`rl.reward_scales`): there β_R1 = 0.1 ≈ 2.3σ.

In [ ]:
scale = nextk_reward_scale(config("nextk_scale", "rl.group_size=8"), prepared, device)
sigma = {"nextk": scale["nextk_group_std"], "topk": scale["topk_group_std"]}
print(scale)
BETA_UNITS = (1.15, 0.5)  # 2.3σ skipped: the item-level runs show it stays near π_ref
betas = {kind: {u: round(u * s, 4) for u in BETA_UNITS} for kind, s in sigma.items()}
betas

## RL with the list reward

Next-K at three β (in σ units 2.3 / 1.15 / 0.5) and Top-K (one softmax, Plackett–Luce lists) at 1.15σ: does conditioning on the earlier positions matter? 2000 steps of 32 contexts × 8 lists; validation every 250 steps on 3000 val users (greedy lists).

In [ ]:
NEXTK = [
    "rl.method=nextk", "rl.batch_size=32", "rl.group_size=8", "rl.steps=2000",
    "rl.eval_every=250", "rl.eval_early_every=100", "rl.eval_early_until=500", "rl.eval_users=3000",
]  # fmt: skip
NEXTK_RUNS = {f"nextk_s{u}": [*NEXTK, f"rl.kl_beta={betas['nextk'][u]}"] for u in BETA_UNITS}
NEXTK_RUNS["topk_s1.15"] = [*NEXTK, "rl.nextk_append=false", f"rl.kl_beta={betas['topk'][1.15]}"]
for name, overrides in NEXTK_RUNS.items():
    train_rl(config(name, *overrides), prepared, device)

In [ ]:
PANELS = [
    ("val_proxy_slate@10", "proxy list reward"),
    ("val_gold_slate@10", "gold list reward"),
    ("val_ndcg@10", "NDCG@10"),
    ("val_recall@10", "Recall@10"),
    ("val_kl_ref", "list KL to π_ref"),
    ("val_gold_ratio@10", "gold item-level ratio@10"),
    ("val_repeat@10", "share of history tracks@10"),
    ("val_artists@10", "distinct artists@10"),
    ("val_length@10", "mean length@10, s"),
    ("val_log_count@10", "mean log10 train count@10"),
]
logs = pl.concat(
    [pl.read_ndjson(runs / n / "log.jsonl").with_columns(run=pl.lit(n)) for n in NEXTK_RUNS],
    how="diagonal_relaxed",
)
logs.write_csv(RESULTS / "nextk_trajectories.csv")
fig, axes = plt.subplots(2, 5, figsize=(25, 8))
for ax, (col, title) in zip(axes.ravel(), PANELS, strict=True):
    for name in NEXTK_RUNS:
        log = logs.filter(pl.col("run") == name)
        ax.plot(log["step"], log[col], marker="o", label=name)
    ax.set(xlabel="step", title=title)
axes[0, 0].legend()
save_figure(fig, RESULTS / "nextk_trajectories.png")
plt.show()
logs.group_by("run").agg(pl.all().sort_by("step").last()).select("run", *[c for c, _ in PANELS])

## Final comparison (val and test, all users)

The greedy list of every policy is its top-10. Also: the per-position search by the slate RM at β = 0.1 / 0.05 / 0 (0 = by the reward alone), Best-of-16 lists of π_ref, the item-level reranker R1 β = 0.1 and the best single item-level policies (RLOO R1 β = 0.05, `distill` β = 0.05) as Top-K lists. Paired bootstrap over users with Holm correction.

In [ ]:
eval_dir = runs / "nextk_eval"
eval_dir.mkdir(parents=True, exist_ok=True)
POLICIES = {
    "pi_ref_topk": (None, False),
    "pi_ref_nextk": (None, True),
    "rl_ratio_b0.05": (runs / "rl_ratio_b0.05" / "policy.pt", False),
    "rl_distill_b0.05": (runs / "rl_distill_b0.05" / "policy.pt", False),
    **{n: (runs / n / "policy.pt", "topk" not in n) for n in NEXTK_RUNS},
}
SEARCHES = {"search_b0.1": (0.1, True), "search_b0.05": (0.05, True), "search_b0": (0.0, True)}
frames = []
for split in ("val", "test"):
    path = eval_dir / f"per_user_{split}.parquet"
    if not path.exists():
        nextk_per_user(
            config("nextk_eval"), prepared, device, split=split, policies=POLICIES,
            searches=SEARCHES, best_of={"bon16": 16},
            extra={"rr_ratio_b0.1": lambda ev: reranker_lists(ev, 0.1)},
        ).write_parquet(path)  # fmt: skip
    frames.append(pl.read_parquet(path))
per_user = pl.concat(frames, how="diagonal_relaxed")
METRICS = [
    "ndcg@10", "recall@10", "proxy_slate@10", "gold_slate@10", "proxy_ratio@10",
    "gold_ratio@10", "kl_ref", "artists@10", "repeat@10", "length@10", "log_count@10",
]  # fmt: skip
table = (
    per_user.group_by("split", "method", maintain_order=True)
    .agg(pl.col(METRICS).mean())
    .sort("split", "ndcg@10", descending=[False, True])
)
table.write_csv(RESULTS / "nextk_comparison.csv")
table

In [ ]:
RL = [n for n in NEXTK_RUNS if n.startswith("nextk")]
PAIRS = (
    [(m, "pi_ref_nextk") for m in RL]
    + [(m, "pi_ref_topk") for m in [*RL, "topk_s1.15"]]
    + [(m, "search_b0.1") for m in RL]
    + [(m, "bon16") for m in RL]
    + [(m, "rr_ratio_b0.1") for m in RL]
    + [(m, "rl_distill_b0.05") for m in RL]
    + [("nextk_s1.15", "topk_s1.15"), ("search_b0.1", "pi_ref_nextk"), ("bon16", "pi_ref_nextk")]
)
boot = bootstrap_table(
    per_user, PAIRS, ["ndcg@10", "recall@10", "proxy_slate@10", "gold_slate@10", "gold_ratio@10",
    "artists@10", "repeat@10"], n_boot=10_000, seed=cfg.seed,
)  # fmt: skip
boot.write_csv(RESULTS / "bootstrap_nextk.csv")
boot.filter(pl.col("metric").is_in(["ndcg@10", "gold_slate@10"])).select(
    "split", "method", "baseline", "metric", "value", "baseline_value", "diff", "ci_low",
    "ci_high", "p_holm",
)  # fmt: skip

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
val = table.filter(pl.col("split") == "val")
for ax, y in zip(axes, ("gold_slate@10", "gold_ratio@10"), strict=True):
    for row in val.iter_rows(named=True):
        marker = "*" if row["method"].startswith(("nextk", "topk_s")) else "o"
        ax.scatter(row["ndcg@10"], row[y], marker=marker, s=90)
        ax.annotate(row["method"], (row["ndcg@10"], row[y]), fontsize=7)
    ax.set(xlabel="NDCG@10 (val)", ylabel=y)
save_figure(fig, RESULTS / "nextk_tradeoff.png")
plt.show()